# 00 · Python 项目结构与 `uv`

**学习目标**

1. 理解 `src/` layout、editable install 与 `import economic_lab` 是如何连起来的。
2. 理解 `pyproject.toml` 中 `[build-system]` 的作用 —— 以及为什么缺少它时 `uv sync` 不会安装你的包。
3. 理解为什么虚拟环境 (virtual environment) 不应放在 iCloud Drive 中。
4. 学会用 `economic_lab.utils.paths` 获取路径，而不是依赖当前工作目录。

本 notebook 不含模型，所有代码都可离线运行。

## 1. `src/` layout

```text
economics/
├── pyproject.toml          # 项目元数据 + 依赖 + 构建配置
├── uv.lock                 # 精确锁定的依赖版本 (提交到 git)
├── src/economic_lab/       # 可导入的 Python 包
│   ├── data/               # 下载、读取、校验、转换、合成数据
│   ├── econometrics/       # 从零实现的估计量 (OLS, ADF, ARDL, ...)
│   ├── visualization/      # 统一绘图风格
│   └── utils/              # 路径等工具
├── scripts/                # 命令行入口 (下载、构建数据集)
├── notebooks/              # 教学 notebook
├── configs/variables.yaml  # 变量登记表 (单位、来源、构造方法)
└── tests/                  # pytest
```

**为什么用 `src/`？** 如果包直接放在项目根目录，Python 会从当前目录导入它 —— 即使你从未安装过它。
测试因此可能在你的电脑上通过，但在别人 `git clone` 之后失败。`src/` layout 强制你 *安装* 包，
测试测的就是用户真正拿到的东西。

## 2. `[build-system]` 与 editable install

原来的 `pyproject.toml` 没有 `[build-system]` 表。对 `uv` 而言，这表示一个 **virtual project**：
`uv sync` 只安装依赖，不安装项目本身。这就是之前需要手动 `uv pip install -e .` 的原因。

现在加入了：

```toml
[build-system]
requires = ["setuptools>=77"]
build-backend = "setuptools.build_meta"

[tool.setuptools.packages.find]
where = ["src"]
```

`uv sync` 会以 **editable** 模式安装 `economic_lab`：site-packages 中只放一个指向 `src/` 的链接，
所以修改源码后无需重新安装。下面验证一下：

In [1]:
import sys
from pathlib import Path

import economic_lab
from economic_lab.utils.paths import processed_data_dir, project_root, raw_data_dir

print("Python      :", sys.version.split()[0])
print("interpreter :", sys.executable)
print("package file:", Path(economic_lab.__file__))
print("project root:", project_root())
print("raw data    :", raw_data_dir())
print("processed   :", processed_data_dir())

Python      : 3.12.13
interpreter : /Users/yuzhezhang/.venvs/economic-lab/bin/python3
package file: /Users/yuzhezhang/Library/Mobile Documents/com~apple~CloudDocs/MyProject/economics/src/economic_lab/__init__.py
project root: /Users/yuzhezhang/Library/Mobile Documents/com~apple~CloudDocs/MyProject/economics
raw data    : /Users/yuzhezhang/Library/Mobile Documents/com~apple~CloudDocs/MyProject/economics/data/econ_data
processed   : /Users/yuzhezhang/Library/Mobile Documents/com~apple~CloudDocs/MyProject/economics/data/processed


`package file` 指向 `src/economic_lab/__init__.py`，说明这是 editable install。

## 3. iCloud Drive 与虚拟环境

项目位于 iCloud Drive 中。iCloud 会把不常用的文件 "evict" 到云端，只留占位符；`.venv` 中有上万个小文件，
`import pandas` 时每个文件都可能触发下载/同步，导致导入极慢（实测：在 iCloud 内的 `.venv` 中启动
Python 超过 2 分钟无响应）。

**解决办法**：让 `uv` 把环境放到 iCloud 之外。在 `~/.zshrc` 中加入：

```bash
# economics 项目的虚拟环境放在 iCloud 之外
export UV_PROJECT_ENVIRONMENT="$HOME/.venvs/economic-lab"
```

之后 `uv sync` / `uv run` 都会使用该路径（实测完整 `uv sync` 约 17 秒）。原来的 `.venv/` 没有被删除，
确认新环境可用后你可以自行删除它。

## 4. 常用命令

| 命令 | 作用 |
|---|---|
| `uv sync` | 按 `uv.lock` 安装核心依赖 + dev 依赖组 + 本项目 (editable) |
| `uv sync --extra interactive` | 额外安装 `ipywidgets`（notebook 滑块） |
| `uv add statsmodels` | 添加依赖并更新 `pyproject.toml` 与 `uv.lock` |
| `uv run pytest` | 在项目环境中运行测试 |
| `uv run python scripts/data_download.py` | 下载可自动获取的数据 |
| `uv run python scripts/build_datasets.py` | 构建 `data/processed/macro_*.parquet` |
| `uv run ruff check src tests` | 代码风格检查 |

可选依赖组（`[project.optional-dependencies]`）：`ml`（XGBoost）、`bayes`（PyMC）、`cge`（Pyomo）、
`interactive`（ipywidgets）。重量级依赖保持可选，核心安装就能保持快速。

In [2]:
from importlib.metadata import version

for pkg in ["numpy", "pandas", "scipy", "statsmodels", "scikit-learn", "matplotlib", "pyarrow"]:
    print(f"{pkg:13s} {version(pkg)}")

numpy         2.5.4
pandas        3.0.6
scipy         1.18.1
statsmodels   0.15.0
scikit-learn  1.9.1
matplotlib    3.11.2
pyarrow       26.0.0


## 5. 包里有什么

下面列出每个模块的第一行 docstring —— 这也是一个好习惯的示范：每个模块开头都用一句话说明它的职责。

In [3]:
import importlib
import pkgutil

for mod in pkgutil.walk_packages(economic_lab.__path__, "economic_lab."):
    m = importlib.import_module(mod.name)
    first = (m.__doc__ or "").strip().splitlines()
    print(f"{mod.name:40s} {first[0] if first else ''}")

economic_lab.data                        Data acquisition, validation and transformation.


economic_lab.data.build                  Build the standardised datasets ``macro_{daily,monthly,quarterly}.parquet``.
economic_lab.data.downloader             Download raw files safely and record their provenance.
economic_lab.data.provenance             Data provenance: checksums and a JSON manifest of every raw file.
economic_lab.data.readers                Parsers that turn raw source files into tidy pandas objects.
economic_lab.data.sources                Registry of raw data sources.
economic_lab.data.synthetic              Synthetic NZ-style macro data with a *known* data-generating process (DGP).
economic_lab.data.transforms             Unit-explicit transformations for macro time series.
economic_lab.data.validation             Data-quality checks for time series.
economic_lab.econometrics                From-scratch econometric estimators (NumPy/SciPy), validated against statsmodels.


economic_lab.econometrics.ardl           Autoregressive distributed lag (ARDL) models and their error-correction form.
economic_lab.econometrics.ols            Ordinary least squares from first principles.
economic_lab.econometrics.timeseries     Time-series foundations: autocorrelation, white-noise tests, unit-root tests.
economic_lab.utils                       
economic_lab.utils.paths                 Project paths.
economic_lab.visualization               
economic_lab.visualization.style         Shared matplotlib style for notebooks and reports.


## 练习

1. **Easy** — 运行 `uv tree --depth 1`，找出 `statsmodels` 依赖了哪些包。
2. **Intermediate** — 在 `src/economic_lab/utils/` 中新建 `timer.py`，写一个上下文管理器打印代码块耗时；
   不重新安装，直接在本 notebook 中 `import` 它（重启 kernel），验证 editable install。
3. **Advanced** — 解释：为什么 `uv.lock` 应该提交到 git，而 `.venv/` 不应该？如果你只提交 `pyproject.toml`，
   一年后别人复现你的结果可能遇到什么问题？

## 参考答案

1. 输出中 `statsmodels` 下的直接依赖包括 `numpy`、`scipy`、`pandas`、`patsy`、`packaging`。
2. 示例：
   ```python
   import time
   from contextlib import contextmanager

   @contextmanager
   def timer(label="block"):
       t0 = time.perf_counter()
       yield
       print(f"{label}: {time.perf_counter() - t0:.3f}s")
   ```
   重启 kernel 后 `from economic_lab.utils.timer import timer` 即可使用，无需重新安装 —— 这就是 editable install。
3. `uv.lock` 记录了每个依赖（包括间接依赖）的精确版本和哈希，保证 "同样的代码 + 同样的依赖"。
   `.venv/` 是平台相关的二进制产物（macOS ARM 的 wheel 在 Linux CI 上无用），且可以由 lock 文件重建。
   只提交 `pyproject.toml` 时，`>=` 约束会解析到一年后的新版本，例如 pandas 的默认行为变化
   （pandas 3 引入 Copy-on-Write 和新的字符串 dtype），数值结果或代码行为都可能改变。